Install Dependencies

In [8]:
!pip install -q yt-dlp openai-whisper
!apt-get -qq install ffmpeg

In [9]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

Download Video

In [10]:
# import yt_dlp

# url = "https://www.youtube.com/watch?v=_f7xDWZzn4c"

# ydl_opts = {
#     "format": "bestvideo+bestaudio/best",
#     "merge_output_format": "mp4",
#     "outtmpl": "video.%(ext)s",
#     "quiet": False,
#  }
# with yt_dlp.YoutubeDL(ydl_opts) as ydl:
#     ydl.download([url])

Extract Audio

In [11]:
# Use FFmpeg to extract audio from the downloaded video file
!ffmpeg -i /content/video.mp4 \
    -ar 16000 \
    -ac 1 \
    /content/audio.wav \
    -y

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

Load Whisper ( to convert audio to text )

In [12]:
import whisper

model = whisper.load_model("base")

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 75.6MiB/s]


In [13]:
import torch
print(torch.cuda.is_available())

True


Generate Transcript

In [14]:
# Transcribe the audio file using the loaded Whisper model
result = model.transcribe(
    "/content/audio.wav",
    word_timestamps=True               # create timestamps
)

In [15]:
# Loop through the first 10 transcription segments
for seg in result["segments"][:10]:
    print(
        f"[{seg['start']:.2f} - {seg['end']:.2f}]",
        seg['text']
    )

[0.00 - 4.32]  There are no subtractions and divisions in our mind.
[5.06 - 7.10]  There is only addition and multiplication.
[7.78 - 11.18]  I will just remove negative thoughts and I will have positive thoughts.
[11.78 - 13.16]  All the best, it's not going to work.
[13.46 - 17.00]  It's just that you need to pay little attention as to how it functions.
[17.54 - 22.14]  You will see there is a distinction between what is you and what you have gathered.
[33.70 - 40.68]  Well, see the way the question is asked and also the way normally it's addressed is
[40.68 - 45.68]  people think there is something called as negative thought and positive thought
[45.68 - 49.80]  they want to remove the negative thoughts and have only positive thoughts.
[50.44 - 56.30]  For such people I would ask them to just experiment for ten, fifteen seconds.


Save as JSON FILE

In [16]:
 # Save the transcript data in JSON format
import json

transcript2 = []

for seg in result["segments"]:
    transcript2.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"]
    })

with open("/content/transcript2.json", "w") as f:
    json.dump(transcript2, f, indent=4)

Download JSON FILE

In [17]:
from google.colab import files

files.download('/content/transcript2.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
!pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 12.6 MB/s eta 0:00:00


In [19]:
import json

file_path = "/content/transcript2.json"

with open(file_path, "r", encoding="utf-8") as f:
    transcript = json.load(f)

print("Transcript loaded successfully!")
print("Number of segments:", len(transcript))

Transcript loaded successfully!
Number of segments: 95


In [20]:
import json
import re
import contractions

In [21]:
FILLERS = [
    "um", "uh", "erm", "ah", "hmm",
    "you know", "like", "sort of", "kind of"
]

def clean_text(text):

    # Expand contractions
    text = contractions.fix(text)

    # Convert to lowercase
    text = text.lower()

    # Remove filler words
    for filler in FILLERS:
        pattern = r"\b" + re.escape(filler) + r"\b"
        text = re.sub(pattern, "", text)

    # Remove unwanted punctuation
    text = re.sub(r"[^\w\s?.!]", "", text)

    # Normalize repeated punctuation
    text = re.sub(r"\?+", "?", text)
    text = re.sub(r"!+", "!", text)
    text = re.sub(r"\.+", ".", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [22]:
cleaned_segments = []

for segment in transcript:

    cleaned_segments.append({
        "start": segment["start"],
        "end": segment["end"],
        "text": clean_text(segment["text"])
    })

print("Cleaning done:", len(cleaned_segments))

Cleaning done: 95


In [23]:
# Save cleaned transcript
with open("cleaned_transcript.json", "w", encoding="utf-8") as f:
    json.dump({"segments": cleaned_segments}, f, indent=4)

print("Transcript cleaned successfully!")

Transcript cleaned successfully!


In [26]:
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
print(type(cleaned_segments))
print(type(cleaned_segments[0]))

In [ ]:
print(any(len(s.get("text", "").strip()) == 0 for s in cleaned_segments))

In [29]:
labels = [
    "Question",
    "Answer",
    "Agreement",
    "Disagreement",
    "Neutral"
]

# Remove empty text segments
valid_segments = [
    seg for seg in cleaned_segments
    if seg.get("text") and seg["text"].strip()
]

print("Original segments:", len(cleaned_segments))
print("Valid segments:", len(valid_segments))

Original segments: 95
Valid segments: 95


In [40]:
texts = [
    s["text"].strip()
    for s in cleaned_segments
    if s.get("text", "").strip()
]

results = classifier(
    texts,
    candidate_labels=labels,
    batch_size=16
)

In [41]:
events = []

for i, seg in enumerate(valid_segments):

    result = classifier(
        seg["text"],
        candidate_labels=labels
    )

    label = result["labels"][0]
    confidence = float(result["scores"][0])

    events.append({
        "start": seg["start"],
        "end": seg["end"],
        "text": seg["text"],
        "label": label,
        "confidence": round(confidence, 3)
    })

    if i % 50 == 0:
        print(f"Processed {i}/{len(valid_segments)}")

Processed 0/95
Processed 50/95


In [42]:
import json

with open("classified_segments.json", "w", encoding="utf-8") as f:
    json.dump(events, f, indent=4)

print("Classification completed!")

Classification completed!


In [43]:
import json

with open("classified_segments.json", "r", encoding="utf-8") as f:
    classified_segments = json.load(f)

print(len(classified_segments))
print(classified_segments[0])

95
{'start': 0.0, 'end': 4.32, 'text': 'there are no subtractions and divisions in our mind.', 'label': 'Answer', 'confidence': 0.258}


In [44]:
detected_events = []

for i, seg in enumerate(classified_segments):

    label = seg["label"]

    # -------------------------
    # Question Answer Detection
    # -------------------------
    if label == "Question":

        if i + 1 < len(classified_segments):

            answer = classified_segments[i + 1]

            detected_events.append({
                "event": "Question-Answer",
                "start": seg["start"],
                "end": answer["end"],
                "question": seg["text"],
                "answer": answer["text"],
                "confidence": (
                    seg["confidence"] +
                    answer["confidence"]
                ) / 2
            })


    # -------------------------
    # Agreement Detection
    # -------------------------
    elif label == "Agreement":

        detected_events.append({
            "event": "Agreement",
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"],
            "confidence": seg["confidence"]
        })


    # -------------------------
    # Disagreement Detection
    # -------------------------
    elif label == "Disagreement":

        detected_events.append({
            "event": "Disagreement",
            "start": seg["start"],
            "end": seg["end"],
            "text": seg["text"],
            "confidence": seg["confidence"]
        })


print("Detected events:", len(detected_events))

Detected events: 81


In [45]:
with open("detected_events.json", "w", encoding="utf-8") as f:
    json.dump(detected_events, f, indent=4)

print("Event detection completed!")

Event detection completed!


In [47]:
import os

print(os.listdir())

['.config', 'video.mp4', 'audio.wav', 'cleaned_transcript.json', 'classified_segments.json', '.ipynb_checkpoints', 'detected_events.json', 'highlight_video.mp4', 'transcript2.json', 'sample_data']


In [37]:
!pip uninstall moviepy -y
!pip install moviepy==1.0.3

Found existing installation: moviepy 1.0.3
Uninstalling moviepy-1.0.3:
  Successfully uninstalled moviepy-1.0.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.3/388.3 kB 9.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for moviepy: filename=moviepy-1.0.3-py3-none-any.whl size=110712 sha256=1aa2e13dff5351e3a6e91e12e8350979bc64f68791dd0d4906a56f2f61d7a054
  Stored in directory: /root/.cache/pip/wheels/df/ba/4b/0917fc0c8833c8ba7016565fc975b74c67bc8610806e930272
Successfully built moviepy


In [48]:
import json
from moviepy.editor import VideoFileClip, concatenate_videoclips


VIDEO_PATH = "video.mp4"
EVENTS_PATH = "detected_events.json"

BUFFER_BEFORE = 8
BUFFER_AFTER = 8

MAX_CLIP_DURATION = 20   # each event max 20 seconds
TARGET_DURATION = 60     # final video


# -----------------------------
# Load video
# -----------------------------

video = VideoFileClip(VIDEO_PATH)

video_duration = video.duration


# -----------------------------
# Load events
# -----------------------------

with open(EVENTS_PATH, "r", encoding="utf-8") as f:
    events = json.load(f)


# Sort by confidence
events = sorted(
    events,
    key=lambda x: x.get("confidence", 0),
    reverse=True
)


selected_clips = []
selected_events = []

total_duration = 0


# -----------------------------
# Select multiple events
# -----------------------------

for event in events:

    if total_duration >= TARGET_DURATION:
        break


    start = max(
        0,
        event["start"] - BUFFER_BEFORE
    )

    end = min(
        video_duration,
        event["end"] + BUFFER_AFTER
    )


    duration = end - start


    # Limit every clip to 20 seconds
    if duration > MAX_CLIP_DURATION:
        end = start + MAX_CLIP_DURATION
        duration = MAX_CLIP_DURATION


    # Avoid duplicate overlapping clips
    overlap = False

    for e in selected_events:

        if abs(start - e["start"]) < 10:
            overlap = True


    if overlap:
        continue


    selected_events.append(event)


    clip = video.subclip(
        start,
        end
    )

    selected_clips.append(clip)

    total_duration += duration



print("Number of clips selected:", len(selected_clips))
print("Final duration:", total_duration)



# -----------------------------
# Merge clips
# -----------------------------

if selected_clips:

    final_video = concatenate_videoclips(
        selected_clips,
        method="compose"
    )


    final_video.write_videofile(
        "highlight_video.mp4",
        codec="libx264",
        audio_codec="aac"
    )


    print("Highlight video created!")

else:
    print("No clips selected")

Number of clips selected: 3
Final duration: 60
Moviepy - Building video highlight_video.mp4.
MoviePy - Writing audio in highlight_videoTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video highlight_video.mp4



Moviepy - Done !
Moviepy - video ready highlight_video.mp4
Highlight video created!


In [39]:
from IPython.display import Video, display

video_path = "highlight_video.mp4"

display(
    Video(
        video_path,
        embed=True,
        width=700
    )
)